In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd

import torchvision
from torchvision import datasets, models, transforms

import numpy as np
import matplotlib.pyplot as plt
import PIL

import math
import os
import random
from tqdm import tqdm

from types import SimpleNamespace
from scipy.fftpack import dct, idct

from art.attacks.evasion import HopSkipJump
from art.estimators.classification import PyTorchClassifier
from art.attacks.inference.membership_inference import MembershipInferenceBlackBox
from art.attacks.inference.membership_inference import LabelOnlyDecisionBoundary
from art.attacks.inference.model_inversion import MIFace
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE

import opacus
from opacus.validators import ModuleValidator
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager

In [ ]:
use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=100, shuffle=False, num_workers=2)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion *
                               planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion*planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.linear = nn.Linear(512*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])

DP-full layer

In [ ]:
def accuracy(preds, labels):
    return (preds == labels).mean()

def train(model, train_loader, optimizer, epoch, device, privacy_engine_input):
    DELTA = 1e-5
    model.train()
    criterion = nn.CrossEntropyLoss()

    losses = []
    top1_acc = []


    for i, (images, target) in enumerate(train_loader):
        images = images.to(device)
        target = target.to(device)

        output = model(images)
        loss = criterion(output, target)
        loss.backward(retain_graph = True)

        preds = np.argmax(output.detach().cpu().numpy(), axis=1)
        labels = target.detach().cpu().numpy()

        acc = accuracy(preds, labels)

        losses.append(loss.item())
        top1_acc.append(acc)

        optimizer.step()
        optimizer.zero_grad()

    epsilon = privacy_engine_input.get_epsilon(DELTA)
    print(
        f"\tTrain Epoch: {epoch} \t"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc@1: {np.mean(top1_acc) * 100:.6f} "
        f"(ε = {epsilon:.2f}, δ = {DELTA})"
    )

def test(model, test_loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    losses = []
    top1_acc = []

    with torch.no_grad():
        for images, target in test_loader:
            images = images.to(device)
            target = target.to(device)

            output = model(images)
            loss = criterion(output, target)
            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = target.detach().cpu().numpy()
            acc = accuracy(preds, labels)

            losses.append(loss.item())
            top1_acc.append(acc)

    top1_avg = np.mean(top1_acc)

    print(
        f"\tTest set:"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc: {top1_avg * 100:.6f} "
    )
    return np.mean(top1_acc)

def accuracy(preds, labels):
    return (preds == labels).mean()

def train_sep(back, head, train_loader, optimizer, epoch, device, privacy_engine_input):
    DELTA = 1e-5
    back.eval()
    head.train()
    criterion = nn.CrossEntropyLoss()

    losses = []
    top1_acc = []


    for i, (images, target) in enumerate(train_loader):
        images = images.to(device)
        target = target.to(device)

        with torch.no_grad():
            images = back(images)

        output = head(images)
        loss = criterion(output, target)
        loss.backward(retain_graph = True)

        preds = np.argmax(output.detach().cpu().numpy(), axis=1)
        labels = target.detach().cpu().numpy()

        acc = accuracy(preds, labels)

        losses.append(loss.item())
        top1_acc.append(acc)

        optimizer.step()
        optimizer.zero_grad()

    epsilon = privacy_engine_input.get_epsilon(DELTA)
    print(
        f"\tTrain Epoch: {epoch} \t"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc@1: {np.mean(top1_acc) * 100:.6f} "
        f"(ε = {epsilon:.2f}, δ = {DELTA})"
    )

def test_sep(back, head, test_loader, device):
    back.eval()
    head.eval()
    criterion = nn.CrossEntropyLoss()
    losses = []
    top1_acc = []

    with torch.no_grad():
        for images, target in test_loader:
            images = images.to(device)
            target = target.to(device)

            images = back(images)
            output = head(images)
            loss = criterion(output, target)
            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = target.detach().cpu().numpy()
            acc = accuracy(preds, labels)

            losses.append(loss.item())
            top1_acc.append(acc)

    top1_avg = np.mean(top1_acc)

    print(
        f"\tTest set:"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc: {top1_avg * 100:.6f} "
    )
    return np.mean(top1_acc)

In [ ]:
MAX_GRAD_NORM =2.4
DELTA = 1e-5
EPOCHS = 25

In [ ]:
model_full_10 = ResNet18()
model_full_10.load_state_dict(torch.load('./checkpoint/final_model.pth')['net'])
model_full_10 = ModuleValidator.fix(model_full_10)
model_full_10 = model_full_10.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_full_10.parameters(), lr=0.1,
                      momentum=0.9, weight_decay=5e-4)

In [ ]:
privacy_engine_full_10 = PrivacyEngine()
model_full_10, optimizer, trainloader = privacy_engine_full_10.make_private_with_epsilon(
    module = model_full_10,
    optimizer = optimizer,
    data_loader = trainloader,
    max_grad_norm = MAX_GRAD_NORM,
    epochs = 25,
    target_epsilon = 0.1,
    target_delta = DELTA,
)

print(f"Using sigma={optimizer.noise_multiplier} and C={MAX_GRAD_NORM}")

In [ ]:
for epoch in tqdm(range(25), desc="Epoch", unit="epoch"):
    train(model_full_10, trainloader, optimizer, epoch + 1, device, privacy_engine_full_10)

In [ ]:
top1_acc = test(model_full_10, testloader, device)

DP- Last layer

In [ ]:
model_last2_10 = ResNet18()
model_last2_10.load_state_dict(torch.load('./checkpoint/final_model.pth')['net'])
resnet_modules = list(model_last2_10.children())
model_last2_10_back = nn.Sequential(*resnet_modules[:-2])
model_last2_10_head = nn.Sequential(*resnet_modules[-2:-1], nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(512, 10))
model_last2_10_head = ModuleValidator.fix(model_last2_10_head)


model_last2_10_back.to(device)
model_last2_10_head.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_last2_10_head.parameters(), lr=0.1,
                      momentum=0.9, weight_decay=5e-4)

In [ ]:
privacy_engine_last2_10 = PrivacyEngine()
model_last2_10_head, optimizer, trainloader = privacy_engine_last2_10.make_private_with_epsilon(
    module = model_last2_10_head,
    optimizer = optimizer,
    data_loader = trainloader,
    max_grad_norm = MAX_GRAD_NORM,
    epochs = 25,
    target_epsilon = 10,
    target_delta = DELTA,
)

print(f"Using sigma={optimizer.noise_multiplier} and C={MAX_GRAD_NORM}")

In [ ]:
for epoch in tqdm(range(25), desc="Epoch", unit="epoch"):
    train_sep(model_last2_10_back, model_last2_10_head, trainloader, optimizer, epoch + 1, device, privacy_engine_last2_10)

In [ ]:
top1_acc = test_sep(model_last2_10_back, model_last2_10_head, testloader, device)

DP-MIA

In [ ]:

class_names = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

classifier = PyTorchClassifier(
    model=model_full_10,
    clip_values=(0, 1),
    loss=nn.CrossEntropyLoss(),
    optimizer=optimizer,
    input_shape=(3, 32, 32),
    nb_classes=len(class_names),
    preprocessing=(0,1)
)

transform_stl = transforms.Compose([
    transforms.Resize((32, 32)),  # STL-10을 CIFAR-10과 같은 크기로 조정
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_stl = datasets.STL10(root='./data', split='train', download=True, transform=transform_stl)


indices_train = torch.randperm(len(trainset))[:100]
indices_test = torch.randperm(len(test_stl))[:100]

subset_train_dataset = torch.utils.data.Subset(trainset, indices_train)
subset_test_dataset = torch.utils.data.Subset(test_stl, indices_test)

train_loader = torch.utils.data.DataLoader(subset_train_dataset, batch_size=100, shuffle=False)
test_loader = torch.utils.data.DataLoader(subset_test_dataset, batch_size=100, shuffle=False)

x_train, y_train = next(iter(train_loader))
x_test, y_test = next(iter(test_loader))

attack = LabelOnlyDecisionBoundary(classifier)
attack.calibrate_distance_threshold_unsupervised(num_samples=400, max_queries=2, top_t=60)

inferred_train = attack.infer(x_train.numpy(), y_train.numpy())
inferred_test = attack.infer(x_test.numpy(), y_test.numpy())

y_true = np.concatenate([np.ones(len(x_train)), np.zeros(len(x_test))])
y_pred = np.concatenate([inferred_train, inferred_test])

auc_score = roc_auc_score(y_true, y_pred)
print(f"AUC score: {auc_score}")

In [ ]:
class_names = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

class CombinedModel(nn.Module):
    def __init__(self, back, head):
        super(CombinedModel, self).__init__()
        self.back = back
        self.head = head
        
    def forward(self, x):
        x = self.back(x)
        #x = x.view(x.size(0), -1)  # 필요한 경우 평탄화
        x = self.head(x)
        return x

combined_model = CombinedModel(model_last2_10_back, model_last2_10_head).to(device)


classifier = PyTorchClassifier(
    model=combined_model,
    clip_values=(0, 1),
    loss=nn.CrossEntropyLoss(),
    optimizer=optimizer,
    input_shape=(3, 32, 32),
    nb_classes=len(class_names),
    preprocessing=(0,1)
)

transform_stl = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_stl = datasets.STL10(root='./data', split='train', download=True, transform=transform_stl)


indices_train = torch.randperm(len(trainset))[:100]
indices_test = torch.randperm(len(test_stl))[:100]

subset_train_dataset = torch.utils.data.Subset(trainset, indices_train)
subset_test_dataset = torch.utils.data.Subset(test_stl, indices_test)

train_loader = torch.utils.data.DataLoader(subset_train_dataset, batch_size=100, shuffle=False)
test_loader = torch.utils.data.DataLoader(subset_test_dataset, batch_size=100, shuffle=False)

x_train, y_train = next(iter(train_loader))
x_test, y_test = next(iter(test_loader))

attack = LabelOnlyDecisionBoundary(classifier)
attack.calibrate_distance_threshold_unsupervised(num_samples=400, max_queries=2, top_t=60)

inferred_train = attack.infer(x_train.numpy(), y_train.numpy())
inferred_test = attack.infer(x_test.numpy(), y_test.numpy())

y_true = np.concatenate([np.ones(len(x_train)), np.zeros(len(x_test))])
y_pred = np.concatenate([inferred_train, inferred_test])

auc_score = roc_auc_score(y_true, y_pred)
print(f"AUC score: {auc_score}")

DP-FC layer

In [ ]:
model_fc_10 = ResNet18()
model_fc_10.load_state_dict(torch.load('./checkpoint/final_model.pth')['net'])
resnet_modules = list(model_fc_10.children())
model_fc_10_back = nn.Sequential(*resnet_modules[:-1])
model_fc_10_head = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(512, 10))
model_fc_10_head = ModuleValidator.fix(model_fc_10_head)


model_fc_10_back.to(device)
model_fc_10_head.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_fc_10_head.parameters(), lr=0.1,
                      momentum=0.9, weight_decay=5e-4)

In [ ]:
privacy_engine_fc_10 = PrivacyEngine()
model_fc_10_head, optimizer, trainloader = privacy_engine_fc_10.make_private_with_epsilon(
    module = model_fc_10_head,
    optimizer = optimizer,
    data_loader = trainloader,
    max_grad_norm = MAX_GRAD_NORM,
    epochs = 25,
    target_epsilon = 0.1,
    target_delta = DELTA,
)

print(f"Using sigma={optimizer.noise_multiplier} and C={MAX_GRAD_NORM}")

In [ ]:
for epoch in tqdm(range(25), desc="Epoch", unit="epoch"):
    train_sep(model_fc_10_back, model_fc_10_head, trainloader, optimizer, epoch + 1, device, privacy_engine_fc_10)

In [ ]:
top1_acc = test_sep(model_fc_10_back, model_fc_10_head, testloader, device)

In [ ]:
class_names = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

class CombinedModel(nn.Module):
    def __init__(self, back, head):
        super(CombinedModel, self).__init__()
        self.back = back
        self.head = head
        
    def forward(self, x):
        x = self.back(x)
        #x = x.view(x.size(0), -1)  # 필요한 경우 평탄화
        x = self.head(x)
        return x

combined_model_fc = CombinedModel(model_fc_10_back, model_fc_10_head).to(device)


classifier = PyTorchClassifier(
    model=combined_model_fc,
    clip_values=(0, 1),
    loss=nn.CrossEntropyLoss(),
    optimizer=optimizer,
    input_shape=(3, 32, 32),
    nb_classes=len(class_names),
    preprocessing=(0,1)
)

transform_stl = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_stl = datasets.STL10(root='./data', split='train', download=True, transform=transform_stl)


indices_train = torch.randperm(len(trainset))[:100]
indices_test = torch.randperm(len(test_stl))[:100]

subset_train_dataset = torch.utils.data.Subset(trainset, indices_train)
subset_test_dataset = torch.utils.data.Subset(test_stl, indices_test)

train_loader = torch.utils.data.DataLoader(subset_train_dataset, batch_size=100, shuffle=False)
test_loader = torch.utils.data.DataLoader(subset_test_dataset, batch_size=100, shuffle=False)

x_train, y_train = next(iter(train_loader))
x_test, y_test = next(iter(test_loader))

attack = LabelOnlyDecisionBoundary(classifier)
attack.calibrate_distance_threshold_unsupervised(num_samples=400, max_queries=2, top_t=60)

inferred_train = attack.infer(x_train.numpy(), y_train.numpy())
inferred_test = attack.infer(x_test.numpy(), y_test.numpy())

y_true = np.concatenate([np.ones(len(x_train)), np.zeros(len(x_test))])
y_pred = np.concatenate([inferred_train, inferred_test])

auc_score = roc_auc_score(y_true, y_pred)
print(f"AUC score: {auc_score}")

In [ ]:
data_full = {
    'Eps': [0.1, 0.5, 1, 5, 10],
    'test_acc': [0.2480, 0.7768, 0.8306, 0.8757, 0.8794],
    'MIA_auc': [0.575, 0.705, 0.755, 0.765, 0.805]
}
df_full = pd.DataFrame(data_full)

data_rec = {
    'Eps': [0.1, 0.5, 1, 5, 10],
    'test_acc': [0.2022, 0.7066, 0.8085, 0.8516, 0.8647], 
    'MIA_auc': [0.52, 0.627, 0.705, 0.75, 0.795]
}
df_rec = pd.DataFrame(data_rec)

plt.figure(figsize=(10, 5))

plt.plot(df_full['Eps'], df_full['test_acc'], marker='o', color='blue', label='Test Accuracy_higher')
for i, txt in enumerate(df_full['test_acc']):
    plt.annotate(f"{txt:.3f}", (df_full['Eps'][i], df_full['test_acc'][i]), textcoords="offset points", xytext=(0,5), ha='center')

plt.plot(df_full['Eps'], df_full['MIA_auc'], marker='o', color='red', label='MIA AUC_higher')
for i, txt in enumerate(df_full['MIA_auc']):
    plt.annotate(f"{txt:.3f}", (df_full['Eps'][i], df_full['MIA_auc'][i]), textcoords="offset points", xytext=(0,5), ha='center')

plt.plot(df_rec['Eps'], df_rec['test_acc'], marker='o', linestyle='--', color='blue', label='Test Accuracy')
plt.plot(df_rec['Eps'], df_rec['MIA_auc'], marker='o', linestyle='--', color='red', label='MIA AUC')


plt.ylim(bottom=min(df_rec['test_acc'].min(), df_rec['MIA_auc'].min()) - 0.05,
            top=max(df_full['test_acc'].max(), df_full['MIA_auc'].max()) + 0.05)

plt.title('Test Accuracy and MIA AUC for the Sub Model with higher gradient norm bound')
plt.xlabel('Epsilon')
plt.ylabel('Value')
plt.xscale('log')
plt.xticks(df_full['Eps'], labels=df_full['Eps'])
plt.grid(True)
plt.legend(loc = 'lower right')

plt.show()